# Stability, review costs and reproduction
**Objectives:** calculate a paired block interval, choose a review threshold without test labels, compare against no action, and interpret a reproduction audit.
**Prerequisites:** notebooks 03, 07 and 10. The analyses are exploratory. A fourth link from the same deployment is not an independent city; retrospective windows do not simulate prospective hyperparameter selection.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
# Match the archived numerical execution context, including neighbor-search kernels.
numeric_threads = threadpool_limits(limits=2)
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd()/'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT/'data/manifest.json').exists()
sys.path.insert(0,str(ROOT/'scripts'))
REVISION=ROOT/'results/revision_v1'


In [ ]:
pred=pd.read_csv(REVISION/'predictions.csv.gz',dtype={'row_id':str},low_memory=False)
paired=pred[pred.case.eq('sensor04_additional_link')&pred.phase.eq('test')&pred.seed.eq(42)]
reference=paired[paired.model.eq('ridge')].set_index('row_id')
alternative=paired[paired.model.eq('mlp')].set_index('row_id').loc[reference.index]
difference=alternative.error.to_numpy()-reference.error.to_numpy()
print('MLP minus Ridge mean absolute error (dB):',difference.mean())


## 1. Resample paired temporal blocks
Negative differences favor the alternative. Resample whole calendar blocks, keeping both models' error for each hour together. Summing errors and counts preserves observation weighting when block sizes differ. This conditions on fitted seed-42 models; it does not repeat hyperparameter selection or sample new deployments.

In [ ]:
def paired_interval(block_days):
    dates=pd.to_datetime(reference.index,utc=True).floor('D')
    group=(dates-dates.min()).days//block_days
    blocks=pd.DataFrame({'difference':difference,'group':group}).groupby('group').difference.agg(['sum','count'])
    totals,counts=blocks['sum'].to_numpy(),blocks['count'].to_numpy()
    rng=np.random.default_rng(42); replicates=[]
    for _ in range(500):
        sampled=rng.integers(len(blocks),size=len(blocks))
        replicates.append(totals[sampled].sum()/counts[sampled].sum())
    low,high=np.quantile(replicates,[.025,.975])
    return dict(block_days=block_days,blocks=len(blocks),difference=difference.mean(),low=low,high=high)
intervals=pd.DataFrame([paired_interval(days) for days in [1,3,7]])
display(intervals)
archived=pd.read_csv(REVISION/'paired_uncertainty.csv')
expected=archived[archived.case.eq('sensor04_additional_link')&archived.model.eq('mlp')].sort_values('block_size')
np.testing.assert_allclose(intervals[['difference','low','high']],expected[['difference','low','high']],atol=1e-12)


**Exercise:** Does a confidence interval excluding zero at one block size prove a practically useful model improvement?

<details><summary>Feedback after your attempt</summary>

no. Dependence assumptions, effect size and costs matter. These intervals omit selection and deployment variability, and multiple exploratory comparisons are not corrected. Inspect the number of blocks and how the interval changes before interpreting it.

</details>

## 2. Select a hypothetical review rule
For false-positive cost 1 and false-negative cost 4, a calibrated probability gives the theoretical threshold 0.2 under zero cost for correct decisions. A second rule selects its threshold on validation. Neither cost was measured in this deployment. The label is a low-SNR received hour, not an incident a review would prevent.

In [ ]:
old=pd.read_csv(ROOT/'results/guide_v3/predictions.csv.gz',dtype={'row_id':str},low_memory=False)
q=old[old.case.eq('classification_sensor02')&old.model.eq('logistic')]
validation=q[q.phase.eq('validation')]; test=q[q.phase.eq('test')]
yv=validation.truth_0.to_numpy().astype(int); pv=validation.prediction_0.to_numpy()
yt=test.truth_0.to_numpy().astype(int); pt=test.prediction_0.to_numpy()
def cost(y,flag,false_negative_cost=4):
    fp=np.sum(flag&(y==0)); fn=np.sum(~flag&(y==1))
    return 100*(fp+false_negative_cost*fn)/len(y)
thresholds=np.unique(np.r_[-1.,pv,1.])
theta=min(thresholds,key=lambda t:(cost(yv,pv>t),np.mean(pv>t),-t))
rows=[]
for name,t in [('validation cost',theta),('probability cost',.2),('flag none',1.),('flag all',-1.)]:
    flag=pt>t
    rows.append(dict(rule=name,threshold=t,flags_per_100=100*flag.mean(),test_cost_per_100=cost(yt,flag)))
display(pd.DataFrame(rows))
saved=pd.read_csv(REVISION/'decision_costs.csv')
expected=saved[saved.case.eq('classification_sensor02')&saved.model.eq('logistic')&saved.rule.eq('validation_cost')&saved.fn_cost.eq(4)].iloc[0]
np.testing.assert_allclose([theta,cost(yt,pt>theta)],[expected.threshold,expected.cost_per_100])


**Exercise:** Explain the result if the selected logistic rule has higher test cost than flagging no hours. May we infer energy savings from the better rule?

<details><summary>Feedback after your attempt</summary>

validation selection does not guarantee improvement on a shifted test period. A credible constant reference is necessary; a model can have predictive structure yet offer no useful operating point for the proposed costs. Energy savings require measured consequences of interventions. Repeat the calculation with costs 1 and 10, while keeping test labels out of threshold selection.

</details>

## 3. Distinguish reproduction claims
The 21 October audit used cached wheels and checksum-verified cached source files. The 23 October audit retrieved all 15 public files and rebuilt the full notebook environment. Both are same-project automation, without independent readers. The newer notebooks ran in-process because external kernel sockets were prohibited; its dependencies came mostly from original cached wheels. Inspect the dated claims separately. The 29 October audit repeats all four experimental rounds; inspect its documented k-NN reference correction. See the repository README for execution instructions.

In [ ]:
audit=json.loads((ROOT/'results/reproduction_check.json').read_text())
display({k:audit[k] for k in ['status','independent_human','remote_retrieval_verified','notebook_environment_rebuilt','scientific_environment_rebuilt']})
display(pd.DataFrame(audit['checks']).T)
remote=json.loads((ROOT/'results/public_retrieval_attempt.json').read_text())
print('Remote retrieval complete:',remote['complete'])
assert audit['status']=='passed' and not audit['remote_retrieval_verified']
current=json.loads((ROOT/'results/public_reproduction_20261023/summary.json').read_text())
print('23 October public-source and notebook check:')
display({k:current[k] for k in ['status','remote_retrieval_verified','notebook_environment_rebuilt','external_kernel_verified','independent_human']})
assert current['status']=='passed' and current['remote_retrieval_verified']
assert not current['independent_human']


In [ ]:
print(json.loads((ROOT/'results/full_reproduction_20261029/summary.json').read_text()))
print(json.loads((ROOT/'results/guide_numeric_revision_20261029.json').read_text())['metric_changes'])


**Adaptation task:** reproduce one module from its public source, record the acquisition mode and source checksum, and have a colleague explain the observation unit, split and supported conclusion. Record any assistance received. Leave unperformed checks explicitly pending; do not replace an unsuccessful remote download with an unlabelled cached copy.